In [0]:
%sql
SELECT 
    current_catalog(), 
    current_database(), 
    current_user();

### Define Ingestion Configuration

In [0]:
INGESTION_CONFIG = [

    {
        "source": "crm",
        "path": "/Workspace/Users/manoranjanpandey836@gmail.com/DataBricks/datasets/data_files/crm_data/cust_info.csv",
        "schema": "bronze",
        "table": "crm_cust_info"
    },
    {
        "source": "crm",
        "path": "/Workspace/Users/manoranjanpandey836@gmail.com/DataBricks/datasets/data_files/crm_data/prd_info.csv",
        "schema": "bronze",
        "table": "crm_prd_info"
    },
    {
        "source": "crm",
        "path": "/Workspace/Users/manoranjanpandey836@gmail.com/DataBricks/datasets/data_files/crm_data/sales_details.csv",
        "schema": "bronze",
        "table": "crm_sales_details"
    }
]


### Ingest Files into Bronze Tables

In [0]:
for item in INGESTION_CONFIG:
    print(f"Ingesting {item['source']} → workspace.bronze.{item['table']}")

    df = (
        spark.read
             .option("header", "true")
             .option("inferSchema", "true")
             .csv(item["path"])
    )

    (
        df.write
          .mode("overwrite")
          .format("delta")
          .saveAsTable(f"workspace.{item['schema']}.{item['table']}")
    )

In [0]:
%sql
drop table workspace.default.current_employees_silver;

In [0]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

# Number of rows - around 10 MB CSV
num_rows = 100000

np.random.seed(42)
random.seed(42)

transaction_types = ["DEBIT", "CREDIT", "TRANSFER", "PAYMENT", "WITHDRAWAL"]
merchant_categories = [
    "Grocery", "Fuel", "Restaurant", "Travel", "Electronics",
    "Healthcare", "Insurance", "Utilities", "Entertainment", "Retail"
]
currencies = ["USD", "EUR", "GBP", "INR", "JPY"]
countries = ["US", "UK", "India", "Germany", "Japan", "Canada", "France"]

start_date = datetime(2023, 1, 1)

data = {
    "transaction_id": [f"TXN{i:08d}" for i in range(1, num_rows + 1)],
    "customer_id": np.random.randint(10000, 99999, num_rows),
    "account_id": np.random.randint(1000000, 9999999, num_rows),
    "transaction_date": [
        start_date + timedelta(minutes=int(x))
        for x in np.random.randint(0, 525600, num_rows)
    ],
    "transaction_type": np.random.choice(transaction_types, num_rows),
    "merchant_category": np.random.choice(merchant_categories, num_rows),
    "amount": np.round(np.random.exponential(scale=250, size=num_rows), 2),
    "currency": np.random.choice(currencies, num_rows),
    "country": np.random.choice(countries, num_rows),
    "account_balance": np.round(np.random.normal(loc=5000, scale=2500, size=num_rows), 2),
    "credit_score": np.random.randint(300, 850, num_rows),
    "is_fraud": np.random.choice([0, 1], size=num_rows, p=[0.97, 0.03])
}

df = pd.DataFrame(data)

# Use Unity Catalog Volume POSIX path
volume_csv_path = "/Volumes/main/default/finance_volume/finance_transactions_10mb.csv"

# Save directly to Volume
df.to_csv(volume_csv_path, index=False)

print("CSV file created successfully!")
print("File saved at:", volume_csv_path)

In [0]:
%fs
ls dbfs:/Volumes/main/default/finance_volume/

In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS main")
spark.sql("CREATE SCHEMA IF NOT EXISTS main.default")
spark.sql("CREATE VOLUME IF NOT EXISTS main.default.finance_volume")

In [0]:
%sql
select * from  
read_files("dbfs:/Volumes/main/default/finance_volume/finance_transactions_10mb.csv",
format => 'csv',
header => true,
inferSchema => true
);


In [0]:
%sql
SELECT 
    current_catalog(), 
    current_database();

### USING SQL CTAS COMMAND

In [0]:
%sql
CREATE OR REPLACE TABLE finance_transactions_delta_sql AS
select * from  
read_files("dbfs:/Volumes/main/default/finance_volume/finance_transactions_10mb.csv",
format => 'csv',
header => true,
inferSchema => true
);

In [0]:
%sql
select * from workspace.default.finance_transactions_delta_sql limit 10;

### USING SPARK

In [0]:
volume_csv_path = "dbfs:/Volumes/main/default/finance_volume/finance_transactions_10mb.csv"
Trans_df = spark.read \
    .csv(volume_csv_path, header=True, inferSchema=True)


In [0]:
Trans_df.write \
    .mode("overwrite") \
    .saveAsTable("Finance_transaction_saprk")

In [0]:
from pyspark.sql.functions import year, month,dayofmonth

Trans_df = (Trans_df
    .withColumn("transaction_year", year("transaction_date")) \
    .withColumn("transaction_month", F.date_format("transaction_date","MMM")) \
    .withColumn("transaction_day",F.date_format("transaction_date","EEE")))


Trans_df.show(10)

In [0]:
Trans_df.groupBy("transaction_type").count().orderBy("count", ascending=False).show()

In [0]:
Trans_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in Trans_df.columns
]).show()